# 03. 이미지 분류(전이학습) & YOLO 객체탐지 — CPU 기준

**GPU 미지원**이므로 작은 모델만 사용합니다. 설치: `pip install torch torchvision ultralytics` (시험 환경에서 설치 가능 여부를 먼저 확인).

데이터는 `practice/data/images/<클래스명>/*.jpg` 구조를 가정합니다.

In [ ]:
import sys; sys.path.append("../common")
from utils import *
import numpy as np, pandas as pd
seed_everything()

## A. 전이학습 (ResNet18 특징 추출 → 로지스틱 회귀)
CPU에서 가장 빠르고 안정적인 방식: 백본은 고정하고 임베딩만 뽑아 가벼운 분류기를 학습합니다.

In [ ]:
import os, glob
IMG_DIR = "../data/images"
assert os.path.isdir(IMG_DIR), "practice/data/images/<클래스>/*.jpg 구조로 데이터를 넣어주세요"
import torch, torchvision
from torchvision import transforms as T
from PIL import Image
torch.set_num_threads(os.cpu_count())

weights = torchvision.models.ResNet18_Weights.DEFAULT
backbone = torchvision.models.resnet18(weights=weights); backbone.fc = torch.nn.Identity(); backbone.eval()
tf = weights.transforms()

files = sorted(glob.glob(f"{IMG_DIR}/*/*.*"))
labels = [os.path.basename(os.path.dirname(f)) for f in files]
@torch.no_grad()
def embed(paths, bs=32):
    out = []
    for i in range(0, len(paths), bs):
        x = torch.stack([tf(Image.open(p).convert("RGB")) for p in paths[i:i+bs]])
        out.append(backbone(x).numpy())
    return np.vstack(out)
with timer("embed"): E = embed(files)
print(E.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
clf = LogisticRegression(max_iter=2000, C=1.0)
s = cross_val_score(clf, E, labels, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1_macro")
print("f1_macro", s.mean().round(4))

## B. YOLO (ultralytics) — 추론 & 소규모 파인튜닝
```python
from ultralytics import YOLO
model = YOLO('yolov8n.pt')            # 가장 작은 모델
res = model.predict('img.jpg', imgsz=640, conf=0.25, device='cpu')
for r in res:
    print(r.boxes.xyxy, r.boxes.conf, r.boxes.cls)

# 파인튜닝: data.yaml(train/val 경로, names) 준비 후
model.train(data='data.yaml', epochs=10, imgsz=416, batch=8, device='cpu', workers=2)
```
- CPU에서는 `imgsz`를 낮추고(320~416) epoch를 적게 잡습니다.
- 가중치(`yolov8n.pt`)는 인터넷이 막힌 환경이면 받아지지 않을 수 있으니 **시험 전 사전 테스트에서 다운로드 가능한지 확인**하세요.
- 결과 csv 포맷(클래스/좌표 형식)은 문제 지문의 요구를 그대로 따릅니다.